# Circuit 6 — Quantum Phase Estimation (3 counting + 1 target = 4 qubits)

**What it does:** Estimates the eigenphase φ of a unitary U = P(φ) applied to a
target qubit. With 3 counting qubits we can represent phases in steps of 1/8.
We use φ = π/4, so the true phase is θ = φ/2π = 1/8 → binary `0.001`,
which is exactly representable and should be read out as `|001⟩` on the
counting register (after noise).

**Circuit structure** (counting qubits q0–q2, target qubit q3):
1. X on q3 — prepare |1⟩, an eigenstate of P(φ) with eigenvalue e^{iφ}
2. H on q0, q1, q2 — uniform superposition on counting register
3. Controlled-U^{2^k} from qk → q3 for k=0,1,2 — encodes the phase into counting register
4. Inverse QFT on q0–q2 — reads out the binary phase estimate

**Two-circuit approach:** Controlled-P(θ) decomposes into P(θ/2)·CNOT·P(-θ/2)·CNOT·P(θ/2).
For φ=π/4, the k=0 rotation P(π/4) is non-Clifford, so `ManyShotRunner` uses a Clifford
stand-in (all P gates replaced by S/S†). The real circuit runs on `TrajectoryBackend`
for the purity pass.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

N_SHOTS      = 2000
N_SHOTS_TRAJ = 400

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
# ── Gate helpers ──────────────────────────────────────────────────────────────

def controlled_phase(c: nq.Circuit, ctrl: int, tgt: int,
                     theta: float, t=None) -> None:
    """Decompose ctrl-P(θ) into 4 standard gates."""
    c.p(ctrl, theta / 2, t=t)
    c.cnot(ctrl, tgt)
    c.p(tgt, -theta / 2)
    c.cnot(ctrl, tgt)
    c.p(tgt, theta / 2)


def controlled_phase_clifford(c: nq.Circuit, ctrl: int, tgt: int,
                               t=None) -> None:
    """Clifford stand-in: replaces P(θ) with S / S†, keeps CNOT structure."""
    c.s(ctrl, t=t)
    c.cnot(ctrl, tgt)
    c.s_dag(tgt)
    c.cnot(ctrl, tgt)
    c.s(tgt)


# ── Shared pass helpers ───────────────────────────────────────────────────────

def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    result_traj = TrajectoryBackend().run(
        circuit, noise_model=noise_config_twirl, n_shots=N_SHOTS_TRAJ, seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities,
                      label, gif_name):
    fig = plot_error_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.many_shot_result = result_many
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_QPE   = 4          # q0, q1, q2 = counting; q3 = target
QPE_PHI = np.pi / 4  # eigenphase of U = P(φ); true θ = 1/8 → binary 0.001


def build_qpe_clifford(phi: float) -> nq.Circuit:
    """
    Clifford stand-in for ManyShotRunner heatmap/GIF.
    All P(θ) gates replaced by S / S†; gate count and CNOT topology preserved.
    """
    c = nq.Circuit(n_qubits=N_QPE, name="qpe_clifford_3bit")

    c.x(3)                       # eigenstate prep
    for q in range(3):           # H on counting register
        c.h(q)

    # Controlled-U^{2^k} stand-ins
    for k in range(3):
        controlled_phase_clifford(c, ctrl=k, tgt=3)

    # Inverse QFT stand-in on q0..q2
    c.swap(0, 2)
    c.h(2)
    controlled_phase_clifford(c, ctrl=1, tgt=2)
    c.h(1)
    controlled_phase_clifford(c, ctrl=0, tgt=1)
    controlled_phase_clifford(c, ctrl=0, tgt=2)
    c.h(0)

    return c


def build_qpe_circuit(phi: float) -> nq.Circuit:
    """
    Real QPE circuit for U = P(phi) on target q3.
    Controlled-U^{2^k} = ctrl-P(phi·2^k). Inverse QFT uses negative phase angles.
    """
    c = nq.Circuit(n_qubits=N_QPE, name="qpe_3bit")

    c.x(3)                       # |1⟩ is eigenstate of P(phi) with e.v. e^{i·phi}
    for q in range(3):           # uniform superposition on counting register
        c.h(q)

    # Controlled-U^{2^k}: each counting qubit k controls P(phi·2^k) on target
    for k in range(3):
        controlled_phase(c, ctrl=k, tgt=3, theta=phi * (2 ** k))

    # Inverse QFT on counting register q0..q2
    # Bit-reversal swap, then H + negative controlled-phase rotations
    c.swap(0, 2)
    c.h(2)
    controlled_phase(c, ctrl=1, tgt=2, theta=-np.pi / 2)
    c.h(1)
    controlled_phase(c, ctrl=0, tgt=1, theta=-np.pi / 2)
    controlled_phase(c, ctrl=0, tgt=2, theta=-np.pi / 4)
    c.h(0)

    return c


circuit_qpe_clifford = fill_idle_with_identities(
    build_qpe_clifford(QPE_PHI), gate_times
)
circuit_qpe = fill_idle_with_identities(
    build_qpe_circuit(QPE_PHI), gate_times
)

print(f"Clifford circuit ops: {len(circuit_qpe_clifford.operations)}")
print(f"Real QPE circuit ops: {len(circuit_qpe.operations)}")

In [ ]:
noise_qpe_clifford = profile.to_noise_model(
    circuit_qpe_clifford, mode="t2", representation="pauli_twirl",
)
noise_qpe = profile.to_noise_model(
    circuit_qpe, mode="t2", representation="pauli_twirl",
)

result_qpe = ManyShotRunner().run(
    circuit_qpe_clifford, n_shots=N_SHOTS, noise_config=noise_qpe_clifford, seed=42,
)
print(f"ManyShotRunner done  zero-error fraction: {result_qpe.zero_error_fraction:.4f}")

rho_qpe, pur_qpe = run_purity_pass(circuit_qpe, noise_qpe, N_QPE, "QPE-3bit")

In [ ]:
# Heatmap and GIF use the Clifford stand-in circuit.
# Purity panel uses rho from the real QPE TrajectoryBackend pass.
visualize_circuit(
    circuit_qpe_clifford, result_qpe, noise_qpe_clifford, rho_qpe, pur_qpe,
    label=f"Phase Estimation (φ=π/4, 3 counting + 1 target)",
    gif_name="qpe_3bit",
)